# Morphological Feature Extraction

**Purpose:** Extracts morphological node statistics (axon, basal dendrite, and apical dendrite
lengths and standard deviations) from eCREST `.json` reconstruction files for each cell type.
These features are used downstream for cell type classification.

**Inputs:**
- `EM_data_published/reconstructions_published/` — eCREST `.json` reconstruction files
- `EM_data_published/data_processed_published/df_type_auto_typed.csv` — cell type assignments

**Outputs:**
- `EM_data_published/data_processed_published/morphology_cat/df_nodes_*.csv` files (one per cell type)

**Used by:** `CellTyping.ipynb`

**Launch requirement:** Run `jupyter lab` from `efish_em/Notebooks_Jupyter/`

# setup

In [1]:
from pathlib import Path
import json
import pandas as pd
from tqdm import tqdm
import cloudvolume
from cloudvolume import CloudVolume
from cloudvolume.exceptions import EmptyFileException
from scipy.optimize import curve_fit

from efish_em import AnalysisCode as efish #only works after the efish_em package has been installed as per README

In [4]:
DATA_ROOT = Path.cwd().parent.parent / 'EM_data_published'
dirpath = DATA_ROOT / 'reconstructions_published'
vx_sizes = [16, 16, 30]

## load cloudvolume and bigquery


In [8]:
efish_cloudvolume = cloudvolume.CloudVolume('gs://fish-ell/roi450um_seg32fb16fb_220930', progress=False)

In [ ]:
# location query run on all base segments in published reconstructions and saved as .parquet and .csv (so can load as either)
seg_bigquery_df = pd.read_parquet(DATA_ROOT / 'base-segs_query_published.parquet')

# get molecular layer plane fit function

In [5]:
neuroglancer_path = Path(settings_dict['save_dir']).parent.parent / 'blender/soma_locations/layer-molecular_annotation.json'

with open(Path(neuroglancer_path), 'r') as myfile: # 'p' is the dirpath and 'f' is the filename from the created 'd' dictionary
    neuroglancer_data = json.load(myfile)

set([item['name'] for item in neuroglancer_data['layers'] if item['type']=='annotation'])

nl_ = 'molecular'
neuroglancer_layer = next((item for item in neuroglancer_data['layers'] if item["name"] == nl_), None)
voxel_sizes = [16,16,30]

vertices = [[p['point'][i]*voxel_sizes[i] for i in range(3)] for p in neuroglancer_layer['annotations']] #[p['point'] for p in neuroglancer_layer['annotations']]#

x_pts = [p[0] for p in vertices]
y_pts = [p[1] for p in vertices]
z_pts = [p[2] for p in vertices]

# Perform curve fitting
popt, pcov = curve_fit(efish.func_planar_curve, (x_pts, z_pts), y_pts)

# Print optimized parameters
print(popt)


[ 2.71956920e+05 -5.43115077e-02 -1.87026179e-01 -3.46153667e-07
  2.31048373e-06  9.59242290e-13 -1.51595014e-11  6.68290149e-07]


# load file list

In [6]:
nodefiles = efish.get_cell_filepaths(dirpath)

# cell types all cells


In [7]:
df_type = pd.read_csv(dirpath / 'metadata/df_type_auto_typed.csv')

In [10]:
cell_type = dict(zip(df_type['id'].values, df_type['cell_type'].values))

# Get cell segment skeleton node locations

## which cells and segments

In [37]:
cell_to_skel = df_type[df_type['cell_type'].isin(['sg2','mg2'])]['id'].values #['mg1','mg2','lg','lf']

In [48]:
segs_toanalyze = {}
dtype = 'axon'
with tqdm(total=len(cell_to_skel)) as pbar:
  for focal_cell_id in cell_to_skel:
      pbar.update(1)
      cell_data = efish.load_ecrest_celldata(nodefiles[str(focal_cell_id)])
      mesh_ids = cell_data['base_segments'][dtype]
      segs_toanalyze[focal_cell_id] = mesh_ids

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 29.07it/s]


## create and save dataframe of segment skeleton info


In [49]:
suff='ax'  # 'ad' 'bd' 'ax'
dfsavepath = Path(settings_dict['save_dir']).parent / 'meta_analysis/morphology_cat/'
dffilename = f'df_nodes_{suff}_type2_updates.csv'


In [50]:
reformatted_dfs = []
for focal_cell_id,mesh_ids in segs_toanalyze.items():
    
    all_verts = []
    with tqdm(total=len(mesh_ids)) as pbar:
        for segment_id in mesh_ids:
            pbar.update(1)
            try:
              skel = efish_cloudvolume.skeleton.get(int(segment_id));
            
              # Iterate over each vertex in the skeleton
              for v in skel.vertices:
            
                  #zero y from molecular layer plane
                  yoffset = efish.func_planar_curve((v[0], v[2]), *popt)
                  y_adj = v[1] - yoffset
                  v = [int(v[0]),int(y_adj),int(v[2])]
            
                  # Append the vertex coordinates to the list
                  all_verts.append(list(v))
            
            except EmptyFileException: # if the segment is too small it doesn't have a skeleton
              # print(f"EmptyFileException: segid {segment_id} is missing. using this segment's location.")
              try:
                  # QUERY = """
                  # SELECT
                  #     cast(objects.id as INT64) as seg_id,
                  #     sample_voxel.x as x,
                  #     sample_voxel.y as y,
                  #     sample_voxel.z as z,
                  # FROM
                  #     `lcht-goog-connectomics.ell_roi450um_seg32fb16fb_220930.objinfo` as objects
                  # WHERE objects.id in {}
                  # """.format('('+str(segment_id)+')')
            
                  df_seg = seg_bigquery_df.query("seg_id==@segment_id") #bigquery_client.query(QUERY).to_dataframe()
                  #zero y from molecular layer plane
                  yoffset = efish.func_planar_curve((df_seg['x'].to_list()[0], df_seg['z'].to_list()[0]), *popt)
                  y_adj = df_seg['y'].to_list()[0] - yoffset
                  v = [df_seg['x'].to_list()[0],int(y_adj),df_seg['z'].to_list()[0]]
                  all_verts.append(list(v))
                  continue
            
              except IndexError:
                  continue
                  
    # all_verts = [v[0] for v in all_verts]
    df = pd.DataFrame(all_verts, columns=['x', 'y', 'z'])
    r = df.describe()
      
    df_flat = r.T.stack().to_frame().T
    df_flat.columns = [f"{suff}_{stat}_{col}" for col, stat in df_flat.columns]
    df_flat['id'] = focal_cell_id

    reformatted_dfs.append(df_flat)

final_df = pd.concat(reformatted_dfs, ignore_index=True)
final_df.set_index('id',inplace=True)
final_df.to_csv(dfsavepath / dffilename)
# #################


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 78/78 [00:19<00:00,  4.02it/s]


## combine updated df with existing csv and resave

In [51]:
fname = '_'.join(dffilename.split('_')[0:-1])
existing_df = pd.read_csv(dfsavepath / f'{fname}.csv')

combined_df = (
    pd.concat([existing_df, final_df.reset_index()])
    .drop_duplicates(subset='id', keep='last')
    .reset_index(drop=True)
)

combined_df.to_csv(dfsavepath / f'{fname}.csv',index=False)